In [ ]:
import torch_directml
import re, sys, os
from pathlib import Path
from dml_fastai_utils import setup_dml, get_local_path
from fastai.vision.all import *

# Initialize DirectML and apply global Vanguard patches
dml = setup_dml()
local_data_path = get_local_path()

## Import Modules

In [ ]:
from fastai.vision.all import *

In [ ]:
import re, sys, osimport loggingfrom pathlib import Pathfrom datetime import datetimeimport numpy as npimport torchimport torch_directmlfrom fastai.vision.augment import _init_mattorch._logging.set_logs(all=logging.INFO)

In [ ]:
print("="*80)print("DIRECTML FULL DEBUG LOG STARTED")print(f"torch version: {torch_directml.torch.__version__}")print(f"Device: {torch_directml.device()}")print(f"GPU: {torch_directml.device_name(0)}")print("="*80)

## Set device to Torch-DirectML detected device

## Load Data to train on

In [ ]:
# 4. Load data (splitting now uses CPU randperm, no crash)path = untar_data(URLs.CAMVID_TINY)files = get_image_files(path / "images")codes = np.loadtxt(path / 'codes.txt', dtype=str)

## Create Fastai DataLoaders

In [ ]:
def label_func(x): return path/'labels'/f'{x.stem}_P{x.suffix}'dls = SegmentationDataLoaders.from_label_func(    path,    bs=8,  # Adjust based on your GPU VRAM (e.g., 32 for lower-end)    fnames=files,    label_func=label_func,    codes=codes,    device=dml,  # Pin batches to DirectML    num_workers=0  # Avoid multiprocessing issues with DirectML)

In [ ]:
# Show one batch to verify labels are correctdls.show_batch(max_n=6)

In [ ]:
test_file = files[0]print("="*80)label = (    f"File: {test_file.name}, "    f"Label: {f'{test_file.stem}_P{test_file.suffix}'}")print(label)

## Create Learner
### Transfer model and everything to Torch-DirectML detected device

In [ ]:
# 5. Create learner on CPU first, then move to DMLlearn = unet_learner(dls, resnet34, metrics=[DiceMulti()]) # resnet18, 34, 50, 101, 152# MixedPrecision disabled globally by setup_dml().to_fp16(enabled=False)   # ← disables mixed precision completelylearn.to(dml)  # Explicitly move to DirectML""

## Start the training/fine-tuning

In [ ]:
print("="*80)learn.fine_tune(epochs=25)print("="*80)

In [ ]:
learn.show_results()

In [ ]:
# interpreter = ClassificationInterpretation.from_learner(learn)

In [ ]:
# interpreter.plot_confusion_matrix()

In [ ]:
# interpreter.plot_top_losses(k=10)